# Dokumentation

Die Dokumentation ist in folgende Abschnitte gegliedert:

1. Datenexploration
2. Datapipeline mit Spark
3. Datenbank (PostgreSQL)
4. Containerisierung mit Docker
5. Interaktive Dashboards mit Streamlit

--------------------------------------------------------------------------------

# Datenexploration und -vorverarbeitung

**Ziel:** Laden und Bereinigen der *NYC HVFHS (High Volume For-Hire Service)*-Daten im Parquet-Format mittels **Apache Spark** und Übertragung in eine **PostgreSQL-Datenbank**.
    


## Datenquelle und Laden

Die Quell-Parquet-Datei wird über:

```python
spark.read.parquet()
```

eingelesen; Spark erkennt Datentypen automatisch.

Jede Zeile repräsentiert eine Fahrt mit Angaben zu:

```text
hvfhs_license_num, PULocationID, DOLocationID, pickup_datetime, dropoff_datetime,
trip_miles, trip_time, base_passenger_fare, driver_pay
```

Durch die verteilte Verarbeitung können auch größere Datenmengen performant analysiert und gefiltert werden.
    


## Erste Exploration und Validierung

Mit:
```python
df.select("hvfhs_license_num").distinct()
df.select("PULocationID", "DOLocationID").distinct()
```
werden eindeutige Anbieter-Codes und Zonen-IDs ermittelt.

Diese Prüfung stellt sicher, dass alle relevanten Dimensionen vorhanden sind, bevor Daten in die Datenbank geschrieben werden.

Über Hilfsfunktionen wie:

```python
get_or_create_providers()
get_or_create_zones()
```

wird geprüft, ob Einträge bereits existieren — fehlende werden automatisch ergänzt.
    


## Datenbereinigung

**Spaltenumbennenung:**
```python
PULocationID → pu_location_id
DOLocationID → do_location_id
```

**Mapping:**  
Anbieter-Codes werden per **UDF** in numerische IDs überführt:

```python
hvfhs_license_num → provider_id
```

**Boolesche Flags:**  
Konvertierung von *Y/N* zu *True/False* für Spalten wie:

```python
shared_request_flag, wav_match_flag
```

**Filterung:**  
Ungültige oder unplausible Fahrten werden ausgeschlossen, z. B.:

```python
trip_miles <= 0
base_passenger_fare <= 0
driver_pay <= 0
```
Diese Schritte standardisieren das Schema und stellen sicher, dass nur qualitativ hochwertige Datensätze weiterverarbeitet werden.
    


## Transformation und Strukturierung

Eine geordnete Spaltenliste wird definiert, um die Zieltabelle **trips** klar zu strukturieren:

```python
trip_cols = [
    "provider_id", "pu_location_id", "do_location_id",
    "pickup_datetime", "dropoff_datetime",
    "trip_miles", "trip_time", "base_passenger_fare", "driver_pay"
]
```

DataFrame-Partitionierung verbessert die Performance beim Schreiben:

```python
df.repartition(args.partitions)
```

Durch Logging ist der gesamte Prozess nachvollziehbar.
    


## Laden in PostgreSQL

Die Verbindung erfolgt über **JDBC**, Zugangsdaten per CLI-Argument.

Bereinigte Datensätze werden batchweise in die Faktentabelle **trips** geschrieben.

Fehlende Informationen zu **Provider** oder **Zonen** werden vorab in den Dimensionstabellen:

```text
providers
taxi_zones
```

ergänzt, um referentielle Integrität zu gewährleisten.
    


## Ergebnis

Entstanden ist ein **bereinigter, konsistenter Datensatz** mit vollständigen Referenzen auf Anbieter- und Zonentabellen.

Alle Spalten folgen einer einheitlichen Namens- und Typkonvention:

```text
BOOLEAN, TIMESTAMP, DECIMAL
```

Das Ergebnis dient als Grundlage für **Analyse- und Dashboard-Komponenten (Streamlit)**, die auf diesen Daten aufbauen.


--------------------------------------------------------------------------------

# Datenpipeline – Architektur und Betrieb

Diese Dokumentation beschreibt die Datenpipeline in diesem Projekt: vom Laden der NYC-Fahrtdaten (Parquet) über Spark nach PostgreSQL bis zur Visualisierung im Streamlit‑Dashboard.

## Überblick
- Komponenten
  - Spark Master/Worker: liest Parquet, transformiert Daten und schreibt per JDBC nach PostgreSQL.
  - PostgreSQL: persistiert Dimensionen (`providers`, `taxi_zones`) und Faktentabelle (`trips`).
  - Streamlit: interaktives Dashboard, das direkt aus PostgreSQL liest.
- Orchestrierung: Docker Compose startet Spark, Postgres und das Dashboard in einem Netzwerk.
- Codepfade
  - Compose/Infra: `Docker/data-pipeline/docker-compose.yml`
  - Dashboard: `Docker/data-pipeline/streamlit_app.py`, `Docker/data-pipeline/dashboard_sections/*`
  - DB‑Connector: `Docker/data-pipeline/data_utilities/database_connector.py`
  - Spark Loader: `Docker/data-pipeline/load_nyc_dataset.py`
  - DB‑Schema: `Docker/data-pipeline/init/database_schema.sql`

## Datenfluss
1) Parquet‑Datei liegt lokal im Container‑Pfad `/app/...`.
2) Spark liest Parquet, bereitet Felder auf, erzeugt fehlende Dimensionen und filtert Ausreißer.
3) Spark schreibt die Spalten in die Tabelle `trips` per JDBC; bei Bedarf werden `providers` und `taxi_zones` ergänzt.
4) Streamlit fragt aus `trips`, `providers`, `taxi_zones` und berechnet KPIs/Charts.

## Lokales Starten (Docker Compose)
- Befehle ausführen im Ordner: `Docker/data-pipeline`
- Start:
  - `docker compose up --build`
  - Services:
    - Spark Master UI: `http://localhost:8080`
    - Spark Worker UI: `http://localhost:8081`
    - Streamlit: `http://localhost:8501`
    - PostgreSQL: `localhost:5432` (User `appuser`, Passwort `group8`, DB `postgres`)

## Daten laden (Spark → Postgres)
- Beispiel (mit Maven‑Paketauflösung):
  - `docker compose exec spark-master spark-submit --packages org.postgresql:postgresql:42.7.8 /app/load_nyc_dataset.py --data-file /app/path/to/data.parquet --pg-host db --pg-user appuser --pg-pass group8`
- Offline‑Variante (treiber JAR liegt im Repo):
  - `docker compose exec spark-master spark-submit --jars /app/postgresql-42.7.8.jar /app/load_nyc_dataset.py --data-file /app/path/to/data.parquet --pg-host db --pg-user appuser --pg-pass group8`
- Wichtige Argumente (`load_nyc_dataset.py`):
  - `--data-file`: Parquet‑Pfad im Container (unter `/app` gemountet)
  - `--pg-host`: innerhalb des Compose‑Netzes `db`, außerhalb `localhost`
  - `--pg-user`, `--pg-pass`, `--pg-db`, `--pg-port`
  - `--partitions` (Default 16), `--batchsize` (Default 1000)

### Annahmen zum Quell‑Schema (Parquet)
- Erwartete Felder (NYC HVFHS‑ähnlich):
  - `hvfhs_license_num`, `PULocationID`, `DOLocationID`
  - Zeitstempel: `request_datetime`, `on_scene_datetime`, `pickup_datetime`, `dropoff_datetime`
  - Metriken/Kosten: `trip_miles`, `trip_time`, `base_passenger_fare`, `tolls`, `bcf`, `sales_tax`, `congestion_surcharge`, `airport_fee`, `tips`, `driver_pay`, `cbd_congestion_fee`
  - Flags: `shared_request_flag`, `shared_match_flag`, `access_a_ride_flag`, `wav_request_flag`, `wav_match_flag` (Y/N)

### Transformationen im Loader
- Provider‑Mapping: hvfhs‑Code → `providers(id)`; unbekannte Codes werden als Name übernommen.
- Zonen: `PULocationID`/`DOLocationID` werden in `taxi_zones` angelegt, falls fehlend.
- Spaltenumbenennung: `PULocationID`→`pu_location_id`, `DOLocationID`→`do_location_id`.
- Flags: Y/N → BOOLEAN; andere Werte → NULL.
- Filter: entfernt Zeilen mit `trip_miles <= 0` oder `base_passenger_fare <= 0` oder `driver_pay <= 0`.
- Schreiben: JDBC nach Tabelle `trips` mit Batchsize/Partitionen.

## Datenmodell (PostgreSQL)
- Datei: `Docker/data-pipeline/init/database_schema.sql`
- Tabellen
  - `providers(id SERIAL PK, provider_name UNIQUE)`
  - `taxi_zones(id SERIAL PK, zone_name)`
  - `trips(id SERIAL PK, provider_id FK, pu_location_id FK, do_location_id FK, ... Kostenfelder, Flags)`
- Typen/Einheiten
  - Zeiten: `TIMESTAMP` (UTC gem. Quelle), Differenzen werden im Dashboard in Minuten berechnet.
  - Distanzen: `trip_miles` in Meilen.
  - Geldbeträge: `DECIMAL(10,2)`.

## Dashboard & DB‑Zugriff
- Einstieg: `Docker/data-pipeline/streamlit_app.py`
- Abschnitte: `Docker/data-pipeline/dashboard_sections/*` mit `render()`
- DB‑Zugriff: `Docker/data-pipeline/data_utilities/database_connector.py`
  - Verbindungsparameter:
    - Host über Env `PG_HOST` (Default: `db` im Compose‑Netz)
    - User: `appuser`, Passwort: `group8`, DB: `postgres`, Port: `5432`

## Validierung
- DB‑Tabellen prüfen:
  - `docker compose exec db psql -U appuser -d postgres -c "\\dt"`
- Sample‑Abfrage:
  - `docker compose exec db psql -U appuser -d postgres -c "SELECT COUNT(*) FROM trips;"`
- UI‑Smoke‑Test: `http://localhost:8501` öffnen und jede Registerkarte einmal laden.

## Häufige Probleme & Lösungen
- JDBC‑Treiber fehlt: Verwende die Offline‑Variante mit `--jars /app/postgresql-42.7.8.jar`.
- Keine Daten im Dashboard: Prüfe, ob `trips` befüllt ist und Zeitfilter nicht leer laufen.
- Shapefile‑Fehler in Geokarten: Stelle sicher, dass alle Dateien in `taxi_zones/` vorhanden sind und `geopandas` installiert ist (im Streamlit‑Image enthalten, siehe `requirements.txt`).
- Verbindung von lokalem Client zu Postgres: Host `localhost`, Port `5432`, User `appuser`, DB `postgres` (Passwort `group8`).

## Erweiterung: Neues Dataset laden
- Parquet muss die oben genannten Felder enthalten oder entsprechend vorprozessiert werden.
- Anpassungen im Loader (`load_nyc_dataset.py`) bei abweichenden Spaltennamen vornehmen.
- Bei neuen Providern: Mapping in `PROVIDER_NAMES` ergänzen (optional – reine Kosmetik).
- Nach Ladevorgang KPIs/Charts in den Sections prüfen; ggf. SQLs um neue Felder erweitern.

## Nützliche Pfade & Referenzen
- Compose: `Docker/data-pipeline/docker-compose.yml:1`
- Loader: `Docker/data-pipeline/load_nyc_dataset.py:1`
- Schema: `Docker/data-pipeline/init/database_schema.sql:1`
- DB‑Connector: `Docker/data-pipeline/data_utilities/database_connector.py:1`
- Dashboard: `Docker/data-pipeline/streamlit_app.py:1`



----------------------------------------------------------------------------------------------------------------------------------------------------------------

# Datenbankschema: Aufbau und Dokumentation

## Überblick

Für das Projekt **„NYC Taxi Data Analysis“** wurde eine **relationale Datenbank** in **PostgreSQL** entwickelt.
Sie bildet die wichtigsten Entitäten und Beziehungen zwischen **Anbietern**, **Stadtzonen** und **Fahrten** ab.
Das Schema (`database_schema.sql`) besteht aus drei Haupttabellen:

- `providers` (Anbieter)
- `taxi_zones` (Stadtzonen)
- `trips` (Fahrten)

Diese Tabellen sind logisch miteinander verknüpft und ermöglichen umfassende Auswertungen zu Fahrverhalten, Kostenstruktur und regionalen Trends.

---

## Tabellenstruktur

### 1. Tabelle: `providers`

| Spalte | Datentyp | Beschreibung |
|---------|-----------|--------------|
| `id` | SERIAL PRIMARY KEY | Eindeutige ID für jeden Anbieter |
| `provider_name` | VARCHAR(32) UNIQUE NOT NULL | Name des Fahrdienstanbieters (z. B. Uber, Lyft) |

**Beschreibung:**
Die Tabelle `providers` speichert alle Anbieter von Fahrdiensten. Jeder Anbieter besitzt eine eindeutige ID, die automatisch generiert wird.
Der Name ist eindeutig und darf nicht mehrfach vorkommen (UNIQUE).

---

### 2. Tabelle: `taxi_zones`

| Spalte | Datentyp | Beschreibung |
|---------|-----------|--------------|
| `id` | SERIAL PRIMARY KEY | Eindeutiger Identifikator der Zone |
| `zone_name` | VARCHAR(64) NOT NULL | Name bzw. Bezeichnung der Stadtzone (z. B. Manhattan, Brooklyn) |

**Beschreibung:**
Die Tabelle `taxi_zones` enthält alle Stadtgebiete, in denen Taxis starten oder enden können.
Sie wird über Fremdschlüssel mit der Tabelle `trips` verknüpft.

---

### 3. Tabelle: `trips`

| Spalte | Datentyp | Beschreibung |
|---------|-----------|--------------|
| `id` | SERIAL PRIMARY KEY | Eindeutige ID jeder Fahrt |
| `provider_id` | INTEGER REFERENCES providers(id) | Verweis auf den Fahrdienstanbieter |
| `pu_location_id` | INTEGER REFERENCES taxi_zones(id) | Startzone (Pickup Location) |
| `do_location_id` | INTEGER REFERENCES taxi_zones(id) | Zielzone (Dropoff Location) |
| `request_datetime` | TIMESTAMP NOT NULL | Zeitpunkt der Fahrtanfrage |
| `on_scene_datetime` | TIMESTAMP | Zeitpunkt, an dem das Taxi am Abholort eintraf |
| `pickup_datetime` | TIMESTAMP NOT NULL | Startzeit der Fahrt |
| `dropoff_datetime` | TIMESTAMP NOT NULL | Ende der Fahrt |
| `trip_miles` | NUMERIC(12,3) | Fahrstrecke in Meilen |
| `trip_time` | NUMERIC(12,3) | Fahrtdauer in Minuten |
| `base_passenger_fare` | DECIMAL(10,2) | Grundpreis der Fahrt |
| `tolls` | DECIMAL(10,2) | Mautgebühren |
| `bcf` | DECIMAL(10,2) | Black Car Fund Gebühr |
| `sales_tax` | DECIMAL(10,2) | Verkaufssteuer |
| `congestion_surcharge` | DECIMAL(10,2) | Stauzuschlag |
| `airport_fee` | DECIMAL(10,2) | Flughafenzuschlag |
| `tips` | DECIMAL(10,2) | Trinkgeld |
| `driver_pay` | DECIMAL(10,2) | Auszahlung an den Fahrer |
| `cbd_congestion_fee` | DECIMAL(10,2) | Zusatzgebühr für das Central Business District |
| `shared_request_flag` | BOOLEAN | Wahr/Falsch – Fahrt wurde als geteilte Fahrt angefragt |
| `shared_match_flag` | BOOLEAN | Wahr/Falsch – Fahrt wurde tatsächlich geteilt |
| `access_a_ride_flag` | BOOLEAN | Wahr/Falsch – Behindertentransport (Access-A-Ride) |
| `wav_request_flag` | BOOLEAN | Wahr/Falsch – Rollstuhlgerechtes Fahrzeug angefragt |
| `wav_match_flag` | BOOLEAN | Wahr/Falsch – Rollstuhlgerechtes Fahrzeug zugeteilt |

**Beschreibung:**
Die Tabelle `trips` enthält sämtliche Fahrten mit zeitlichen Angaben, Preisen, Gebühren und zusätzlichen Kennzeichen.
Sie verknüpft Anbieter (`providers`) und Stadtzonen (`taxi_zones`) über Fremdschlüssel und bildet die zentrale Datengrundlage für alle Analysen.
So können Auswertungen zu Preisstrukturen, Fahrverhalten, Fahrdauer und Servicearten durchgeführt werden.

---

## Umsetzung mit Docker Compose

Die Datenbank wird als Service `db` in einem **PostgreSQL-Container** betrieben.
Beim **ersten Start** des Containers werden automatisch alle SQL-Skripte aus dem Verzeichnis `./init/` (z. B. `database_schema.sql`) ausgeführt.
Dies geschieht, weil der Ordner als `docker-entrypoint-initdb.d` gemountet ist – ein Standardmechanismus von PostgreSQL-Dockerimages.

**Wichtige Hinweise:**
- Die SQL-Skripte werden nur beim **ersten Start** ausgeführt, wenn das Datenverzeichnis leer ist (neues Volume).
- Die Ausführung erfolgt gegen die in `POSTGRES_DB` definierte Datenbank (aktuell: `postgres`).
- Wenn eine eigene Datenbank gewünscht ist, z. B. `nyc_taxi`, kann dies durch Anpassung der Umgebungsvariablen erfolgen:
  ```yaml
  - POSTGRES_DB=nyc_taxi

--------------------------------------------------------------------------------

## Projekt Containerisierung
In diesem Projekt wird Docker eingesetzt, um die gesamte Infrastruktur reproduzierbar, portabel und einfach verwaltbar zu machen. Data-Engineering Workflows erfordern oft mehrere miteinander verbundene Systeme, etwa Spark für verteilte Datenverarbeitung, PostgreSQL als Datenbank und Streamlit für das Dashboard. Mit Docker lassen sich diese Komponenten als isolierte Container ausführen, ohne Konflikte zwischen Bibliotheken oder Umgebungen.
Der entscheidende Vorteil liegt darin, dass alle Projektmitglieder und auch Produktionssysteme mit exakt derselben Umgebung arbeiten, und das ganz unabhängig vom lokalen Betriebssystem oder installierten Tools.



### Docker Compose
Docker Compose dient als Orchestrierungswerkzeug, um mehrere Container gleichzeitig zu starten und zu verbinden. Statt jeden Dienst einzeln zu starten beschreibt die docker-compose.yml das gesamte System in einem einzigen YAML-File.
So entsteht eine kleine, reproduzierbare Datenplattform, in der Spark, PostgreSQL und das Streamlit-Dashboard automatisch miteinander kommunizieren können.

<br/>
Der Aufbau von *docker-compose.yml* sieht wie folgt aus:

```yaml
version: "3.8"

services:
  spark-master:
    image: spark:3.5.7-scala2.12-java17-python3-ubuntu
    container_name: spark-master
    environment:
      - SPARK_NO_DAEMONIZE=true         # keep process in foreground
      - SPARK_PUBLIC_DNS=localhost      # optional: nicer UI links
    user: root
    command: >
        /bin/bash -c "apt-get update && pip install --upgrade pip && pip install psycopg2-binary && /opt/spark/sbin/start-master.sh"

    ports:
      - "7077:7077"
      - "8080:8080"
    volumes:
      - .:/app
    networks: [spark-net]

  spark-worker-1:
    image: spark:3.5.7-scala2.12-java17-python3-ubuntu
    container_name: spark-worker-1
    depends_on:
      - spark-master
    environment:
      - SPARK_NO_DAEMONIZE=true
    user: root
    command: ["/opt/spark/sbin/start-worker.sh", "spark://spark-master:7077"]
    ports:
      - "8081:8081"
    volumes:
      - .:/app
    networks: [spark-net]

  db:
    image: postgres:16
    container_name: nyc_tlc_postgres16
    restart: unless-stopped
    environment:
      - POSTGRES_USER=appuser
      - POSTGRES_PASSWORD=group8
      - POSTGRES_DB=postgres
    ports:
      - "5432:5432"
    volumes:
      - pgdata:/var/lib/postgresql/data
      - ./init:/docker-entrypoint-initdb.d:ro
    networks: [spark-net]

  streamlit:
    build:
      context: .
      dockerfile: Dockerfile.streamlit
    container_name: streamlit-dashboard
    depends_on:
      - db
    ports:
      - "8501:8501"
    volumes:
      - .:/app
    networks: [spark-net]

volumes:
  pgdata:

networks:
  spark-net:
    driver: bridge

#### Übersicht der Docker Services


##### **Spark-Master**

Der Spark-Master ist das Steuerzentrum des Spark-Clusters. Er verwendet das Image *spark:3.5.7-scala2.12-java17-python3-ubuntu* und startet über das Kommando:

`/opt/spark/sbin/start-master.sh`

Dabei werden Umgebungsvariablen gesetzt, um die Ausführung im Vordergrund zu halten (SPARK_NO_DAEMONIZE=true) und eine lokale Oberfläche auf Port 8080 bereitzustellen. Das Volume-Mapping (.:/app) bindet den Projektordner in den Container ein, damit Spark auf die lokalen Skripte und Datendateien zugreifen kann.


##### **Spark-Worker-1**

Der Worker-Container führt die eigentlichen Rechenoperationen aus. Er verbindet sich über
`spark://spark-master:7077`
mit dem Master und führt die von ihm zugewiesenen Tasks aus.
Durch das gleiche Volume-Mapping kann er auf dieselben Skripte und Daten zugreifen.

##### **PostgreSQL-Datenbank**
Die Datenbank `db` basiert auf dem offiziellen Image `postgres:16`. Sie dient als persistente SQL basierte Speicherstruktur für transformierte Daten.
Über die Umgebungsvariablen wird ein Benutzer (appuser) und ein Passwortdefiniert.
Das Volume `pgdata` sorgt dafür, dass die Datenbankdaten auch nach einem Neustart erhalten bleiben.
Die Initialisierungsdateien im Ordner ./init werden beim ersten Start automatisch ausgeführt, um Tabellen oder Schemas anzulegen.

##### **Streamlit**
Der Streamlit-Service bildet die Visualisierungsebene des Projekts. Er wird über ein eigenes *Dockerfile* gebaut, damit die Python-Abhängigkeiten gezielt installiert und die Projektdateien in den Container kopiert werden können.



### Streamlit Konfiguration
Während Spark und PostgreSQL über offizielle Images laufen, benötigt Streamlit eine auf das Projekt zugeschnittene Python-Umgebung, die nicht in einem Standardimage enthalten ist. Ein dediziertes Dockerfile dient dazu, eine eigenständige und reproduzierbare Laufzeitumgebung für das Streamlit-Dashboard zu schaffen.
##### **Dockerfile von Streamlit** *(Dockerfile.streamlit)*



```yaml
FROM python:3.10-slim

WORKDIR /app
COPY streamlit_app.py .
COPY requirements.txt .
COPY data_utilities/ ./data_utilities/
COPY dashboard_sections/ ./dashboard_sections/
COPY taxi_zones/ ./taxi_zones/

RUN pip install --upgrade pip
RUN pip install -r requirements.txt

EXPOSE 8501

CMD ["streamlit", "run", "streamlit_app.py", "--server.port=8501", "--server.address=0.0.0.0"]
```

### Importprozess für Spark
Anstatt die Rohdaten direkt in PostgreSQL zu importieren, nutzt das Projekt Spark, um große Datenmengen effizient zu verarbeiten und lädt die Daten nach erfolgreicher Transformation in die Datenbank.
Nach dem Starten des Docker Clusters muss somit der Datensatz über folgenden Docker Befehl in die Datenbank geladen werden:
```yaml
docker exec spark-master /opt/spark/bin/spark-submit \
  --master spark://spark-master:7077 \
  --driver-memory 2g \
  --conf "spark.executor.memory=2g" \
  --jars /app/postgresql-42.7.8.jar \
  /app/load_nyc_dataset.py \
  --data-file /app/fhvhv_tripdata_2025-07.parquet \
  --pg-host db --pg-port 5432 --pg-db postgres --pg-user appuser --pg-pass group8 \
  --partitions 16 --batchsize 1000
```
Dieser Befehl führt innerhalb des Spark-Master Containers das Python-Skript `load_nyc_dataset.py` aus. Das Skript liest den NYC TLC Datensatz ein, bereitet ihn mit Spark vor und lädt die Daten anschließend in die Datenbank.




--------------------------------------------------------------------------------

# Streamlit Dokumentation


## 1. Code-Struktur

Folgende Directories in data-pipeline sind ausschließlich für Streamlit verantwortlich:
 ```
data-pipeline/
├── streamlit_app.py
├── data_utilities/
│   └── database_connector.py
├── dashboard_sections/
│   ├── section_diego.py
│   ├── section_daniel.py
│   ├── section_duong.py
│   ├── section_negar.py
│   └── section_tugba.py
└── taxi_zones/
    ├── taxi_zones.shp
    ├── taxi_zones.shx
    ├── taxi_zones.dbf
    └── ... (weitere Shapefile-Komponenten)
```

1.  **`streamlit_app.py`**: Die Streamlit "Main"-Datei, sie importiert die Dashboardsections (die verschiedenen Userstories) und strukturert sie in Tabs. Die Inhalte werden mit der `render()`-Funktion der Dashboardsections angezeigt.
2.  **`data_utilities/database_connector.py`**: Stellt Funktion bereit die sich mittels SQLAlchemy-Engine mit der Postgres-Datenbank verbindet.
3.  **`dashboard_sections/`**: Jede `.py`-Datei in diesem Ordner repräsentiert eine User Story. Jede Datei enthält eine `render()`-Funktion, die von der `streamlit_app.py` aufgerufen wird.
4.  **`taxi_zones/`**: Enthält die Shape-Datei für die Interaktive Karte (genutzt von `section_diego.py`).

## 2. Datenabruf & Caching


### 2.1 Datenabruf in einer Sektion
Jede Sektion importiert die SQLAlchemy-Engine und nutzt sie in Kombination mit pandas.read_sql_query, um Daten zu laden.

Folgende Schritte finden statt:

1. SQLAlchemy-Engine angeben

2. SQL-Query definieren

3. Query und Engine an pandas.read_sql_query übergeben

*Beispiel (aus section_diego.py):*
```
import pandas as pd
from data_utilities import database_connector

def load_market_share():
    engine = database_connector.get_sqlalchemy_engine()
    query = """
            SELECT p.provider_name, COUNT(*) AS total_trips
            FROM trips t JOIN providers p ON t.provider_id = p.id
            GROUP BY p.provider_name
            """
    df = pd.read_sql_query(query, engine) #
    return df
```

### 2.2 Datenverarbeitung & Caching
Die Performance des Dashboards hängt entscheidend davon ab, wie oft und wie viele Daten aus der Datenbank geladen werden. Bei den knapp 20Mio Einträgen im Quell-Dataset ist Caching unerlässlich. Dazu verwenden wir den Streamlit-Decorator @st.cache_data. Dieser "merkt" sich das und die Daten müssen nicht ständig neu geladen werden.

```
@st.cache_data(show_spinner="Lade Preisvergleichsdaten...", ttl=600)
def load_price_distribution_data(distance_range: tuple):
    ...
```

**Parameter:**
`show_spinner`: Zeigt eine Ladekringel in der UI an während die Funktion und die SQL-Abfrage läuft.
`ttl`: (Time-To-Live) Weist Streamlit an, den Cache-Eintrag nach 600 Sekunden (10 Minuten) zu invalidieren. Das Dashboard wird so zur Sicherheit nach den 10 Minuten aktualisiert.

Parameter-Tracking: Der Cache ist an die Argumente der Funktion gebunden. load_price_distribution_data((3.0, 10.0)) wird nur einmal ausgeführt. Ändert der User den Slider auf (4.0, 11.0), wird die Funktion einmal neu ausgeführt und das Ergebnis für diese neuen Parameter gespeichert.

#### 2.3 Aggregation und Sampling

Aggregation in SQL (Bevorzugt): Für die meisten Diagramme (Heatmap, Barchart, KPIs) wird die Aggregation (AVG, GROUP BY, COUNT) direkt in der PostgreSQL-Datenbank durchgeführt. So werden nur wenige hundert Zeilen an Python/Pandas gesendet.

Sampling in SQL (Für Rohdaten-Plots): Für Visualisierungen, die Rohdaten benötigen (Boxplot, Scatter-Plot), macht eine Aggregation keinen Sinn. Stattdessen werden Samples gezogen, die die knapp 20mio einträge repräsentieren sollen (z.B in section_diego.py Zeile 20)

#### 2.4 Transformation in Pandas
Nach dem Laden werden die Daten in Pandas weiterverarbeitet.
Die verschiedenen Plots werden dann in Streamlit auf Basis der resultierenden Pandas-Dataframes dargestellt.

#### 2.5 Kartendaten
Die Kartendaten liegen als Shapefile vor und können mithilfe von Geopandas eingelesen werden. Dieses speichert die Geometrien als Koordinatenfolgen ab, genau wie Pandas auch als Dataframe bzw. Geodataframe.

# 3. Dashboardstruktur


### 3.1 streamlit_app.py
Diese Datei dient nur als Container. Sie nutzt st.tabs, um die Sektionen der einzelnen Userstories voneinander zu trennen. Jede Sektion wird in einen eigenen Tab geladen und ihre render()-Funktion aufgerufen.


### 3.2 Interne Sektions-Struktur

Das am häufigsten verwendeten Muster zur Anordnung sind st.tabs und st.columns. Damit lassen sich Inhalte in derselben Zeile in verschiedenen Spalten anzeigen.
Beispielsweise KPIs (st.metric) oder natürlich auch Plots. Tabs und Spalten werden wie folgt definiert:

```
tab_overview, tab_detail, tab_docs = st.tabs([
    "Übersicht",
    "Anbieter-Detail",
    "Dokumentation"
])

with tab_overview:
    # Code für die Übersichts-KPIs...
with tab_detail:
    # Code für die Detail-Analyse...
```
```
row2_col1, row2_col2 = st.columns(2)
row3_col1, row3_col2 = st.columns(2)

with row2_col1:
    # Code für den Boxplot...
with row2_col2:
    # Code für den Scatter-Plot...
```

In den Tabs und Sections können dann die Inhalte, also KPIs, Plots, Interaktive Slider, Radiobuttons etc. eingefügt werden.